In [6]:
import subprocess,sys,os,warnings,glob
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
warnings.filterwarnings("ignore")
def install_deps():

    base = ["transformers==4.46.3", "peft==0.13.2", "trl==0.12.2",

            "accelerate==1.1.1", "datasets==2.21.0"]

    print("[1/3] Installing transformers/peft/trl/accelerate/datasets ...")

    subprocess.check_call(

        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *base],

        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,

    )
    candidates = [

            ("--pre", "bitsandbytes"),            # actively built against current CUDA

            ("",     "bitsandbytes==0.46.0rc1"),  # most recent 0.46 release candidate

            ("",     "bitsandbytes==0.45.2"),

            ("",     "bitsandbytes==0.45.0"),
        ]

    print("installing bitsandbytes")

    for flags,spec in candidates:
        try:
            cmd=[sys.executable,"-m","pip","install","-q","--upgrade"]
            if flags:
                cmd.append(flags)
            cmd.append(spec)
            subprocess.check_call(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
            check = (

                    "import bitsandbytes as bnb, os, glob;"

                    "libs = glob.glob(os.path.join(os.path.dirname(bnb.__file__),"

                    "                              'libbitsandbytes_cuda*.so'));"

                    "assert libs, f'no bnb cuda lib (got {libs})';"

                    "print('bnb ' + bnb.__version__ + ' libs: ' + str(libs))"

                )
            subprocess.check_call([sys.executable,"-c",check])
            print(f"->{spec}OK")
            break
        except Exception as e:

            print(f"   !! {spec} failed: {type(e).__name__}: {e}")

    else:

        raise SystemExit(

            "Could not install a working bitsandbytes. Try a fresh GPU session "

            "and/or set Docker image to 'Standard' (without RAPIDS) on Kaggle."

        )

    print("[3/3] Pinning triton to a compatible version ...")

    subprocess.check_call(

        [sys.executable, "-m", "pip", "install", "-q", "triton==3.0.0"],

        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,

    )

install_deps()

import transformers, peft, trl, accelerate, datasets, bitsandbytes, torch

print("\n=== Version check ===")

print("transformers :", transformers.__version__)

print("peft         :", peft.__version__)

print("trl          :", trl.__version__)

print("accelerate   :", accelerate.__version__)

print("datasets     :", datasets.__version__)

print("bitsandbytes :", bitsandbytes.__version__)

print("torch        :", torch.__version__)

print("CUDA?        :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU count    :", torch.cuda.device_count())

    print("GPU[0]       :", torch.cuda.get_device_name(0))

bnb_dir = os.path.dirname(bitsandbytes.__file__)

bnb_libs = glob.glob(os.path.join(bnb_dir, "libbitsandbytes_cuda*.so"))

print("bnb libs     :", bnb_libs)

assert bnb_libs, "bitsandbytes has no CUDA lib -- QLoRA will fail"


[1/3] Installing transformers/peft/trl/accelerate/datasets ...
installing bitsandbytes
->bitsandbytesOK
[3/3] Pinning triton to a compatible version ...

=== Version check ===
transformers : 4.46.3
peft         : 0.13.2
trl          : 0.12.2
accelerate   : 1.1.1
datasets     : 2.21.0
bitsandbytes : 0.49.2
torch        : 2.11.0+cu128
CUDA?        : True
GPU count    : 1
GPU[0]       : Tesla T4
bnb libs     : ['/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda122.so', '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda124.so', '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda121.so', '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda123.so', '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda129.so', '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda125.so', '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda130.so', '/usr/l

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig,prepare_model_for_kbit_training,PeftModel
from trl import SFTTrainer,SFTConfig


In [8]:
raw=load_dataset("tatsu-lab/alpaca",split='train')
def format_row(row):
    if row["input"]:
        text=f"<s>[INST]{row['instruction']}\n{row['input']}[/INST]{row['output']}</s>"
    else:
        text=f"<s>[INST]{row['instruction']}[/INST]{row['output']}</s>"
    return{"text":text}
dataset=raw.map(format_row,remove_columns=raw.column_names)
dataset=dataset.select(range(1000))


Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [11]:
MODEL_NAME="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR="output"
MAX_STEPS=200

In [12]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"":0},
    trust_remote_code=True,
)
model.config.use_cache=False

if hasattr(model, "module"):

    print("Unwrapping DataParallel/DistributedDataParallel layer ...")

    model = model.module

assert all(

    p.device.type == "cuda" and p.device.index == 0

    for p in model.parameters()

), f"Model parameters are on multiple devices: {set((p.device.index, p.device.type) for p in model.parameters())}"

print(f"Model fully on cuda:0")


model = prepare_model_for_kbit_training(model)

print("Model loaded in 4-bit")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model fully on cuda:0
Model loaded in 4-bit


In [13]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"
print("Tokenizer loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer loaded


In [14]:
peft_config=LoraConfig(
    r=32,lora_alpha=16,lora_dropout=0.05,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    bias="none",task_type="CAUSAL_LM",
)

In [15]:
import os as _os2
for _k in ("LOCAL_RANK","RANK","WORLD_SIZE","MASTER_ADDR","MASTER_PORT"):
    _os2.environ.pop(_k,None)

sft_config=SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    fp16=True,
    max_steps=MAX_STEPS,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    max_seq_length=512,
    dataset_text_field="text",
    packing=False,
)
trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,

)
for name,module in trainer.model.named_modules():
    if "norm" in name:
        module=module.to(torch.float32)
trainable=sum(p.numel() for p in model.parameters()if p.requires_grad)
total=sum(p.numel() for p in model.parameters())

print(f"Trainer ready — {trainable:,}/{total:,} ({100*trainable/total:.2f}%)")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


Trainer ready — 9,011,200/624,617,472 (1.44%)


In [16]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=200, training_loss=1.4335198974609376, metrics={'train_runtime': 361.3287, 'train_samples_per_second': 8.856, 'train_steps_per_second': 0.554, 'total_flos': 3072016556654592.0, 'train_loss': 1.4335198974609376, 'epoch': 3.2})

In [17]:
os.makedirs(OUTPUT_DIR,exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('output/tokenizer_config.json',
 'output/special_tokens_map.json',
 'output/tokenizer.model',
 'output/added_tokens.json',
 'output/tokenizer.json')

In [21]:
base_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME, torch_dtype=torch.float16,

    device_map={"": 0},  # single-GPU for inference too

)

In [22]:
fine_tuned=PeftModel.from_pretrained(base_model,OUTPUT_DIR)
fine_tuned.eval()


PeftModel(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

In [26]:
def ask(instruction, input_text=""):
    prompt = (f"<s>[INST]{instruction}\n{input_text}[/INST]"
              if input_text else f"<s>[INST]{instruction}[/INST]")
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned.device)
    with torch.no_grad():
        output = fine_tuned.generate(
            **inputs, max_new_tokens=250, pad_token_id=tokenizer.eos_token_id,
            temperature=0.7, do_sample=True,  # ← fixed spelling
        )
    return tokenizer.decode(output[0], skip_special_tokens=True).split("[/INST]")[-1].strip()

In [27]:
for q in [

    "What skills are most important for an ML Engineer role?",

    "Give me 3 tips for a strong resume.",

    "What questions should I prepare for a Python developer interview?",

    "Explain overfitting in simple terms.",

]:

    print("=" * 60)

    print(f"Q: {q}")

    print("-" * 60)

    print(f"A: {ask(q)}\n")

print("Your AI Interview Coach is working!")


Q: What skills are most important for an ML Engineer role?
------------------------------------------------------------
A: An ML engineer should have a strong understanding of machine learning concepts, algorithms, programming languages, and data structures. They should also have experience working with various datasets, algorithms, and libraries. Additionally, they should have strong communication and analytical skills and be able to understand and solve complex problems. They should also be able to work well in a team and collaborate with other engineers, data scientists, and stakeholders. Finally, they should be able to troubleshoot issues and solve problems efficiently and effectively. 

Overall, an ML engineer should have a deep understanding of the field and be able to translate that understanding into practical solutions. They should also be able to communicate their insights and solve problems with clarity and efficiency.

In summary, an ML engineer should have a strong underst